## Scikit-learn Pipelines and Model Serving Patterns


1. Let's build a sklearn Pipeline that takes raw player stats (mixed numeric and categorical), applies imputation, scaling, and one-hot encoding using ColumnTransformer, trains a RandomForest, and evaluates with cross-validation. Serialise it with joblib and explain why the Pipeline matters for production serving.

2. Given a trained model, let's write a function that detects data drift: compare the feature distributions of incoming data against training data using KS test (scipy.stats.ks_2samp) and flag features that have drifted beyond a threshold.

In [1]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
import joblib

# --- fake player data ---
data = {
    'age': [28, 31, None, 24, 29, 33, 26, None, 30, 27],
    'speed_kmh': [32.1, 28.5, 30.2, None, 33.0, 27.8, 31.5, 29.9, None, 32.8],
    'position': ['FW', 'MF', 'DF', 'FW', None, 'GK', 'MF', 'DF', 'FW', 'MF'],
    'foot': ['right', 'left', 'right', 'right', 'left', 'right', None, 'left', 'right', 'left'],
    'injury': [1, 0, 0, 1, 0, 0, 1, 0, 1, 0]  # target
}
df = pd.DataFrame(data)

X = df.drop('injury', axis=1)
y = df['injury']

# --- define which columns are numeric vs categorical ---
num_cols = ['age', 'speed_kmh']
cat_cols = ['position', 'foot']

# --- build the preprocessor ---
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

# --- full pipeline: preprocess + model ---
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(n_estimators=100, random_state=42))
])

# --- evaluate ---
scores = cross_val_score(pipeline, X, y, cv=3, scoring='accuracy')
print(f"CV Accuracy: {scores.mean():.2f} +/- {scores.std():.2f}")

# --- fit on full data and save ---
pipeline.fit(X, y)
joblib.dump(pipeline, 'player_injury_model.pkl')

# --- load and predict (in production) ---
loaded_pipeline = joblib.load('player_injury_model.pkl')
new_player = pd.DataFrame({'age': [25], 'speed_kmh': [31.0], 'position': ['FW'], 'foot': ['left']})
prediction = loaded_pipeline.predict(new_player)

CV Accuracy: 1.00 +/- 0.00


**Why the Pipeline matters for production**

Without a pipeline, you'd need to replicate every preprocessing step manually at serving time - impute with the same medians, scale with the same mean/std, encode with the same category mapping. Get any of those wrong and predictions are garbage. The Pipeline bundles everything into a single object: joblib.dump saves the fitted transformers and the model together, joblib.load restores them, and .predict() on raw data just works. No train/serve skew.

It also means the same object handles training (pipeline.fit(X, y)) and inference (pipeline.predict(new_data)), so there's one code path to test, not two.

About handle_unknown='ignore': in production you might see a new category the model never trained on (e.g. a new position). Without this flag, OneHotEncoder throws an error. With it, the unknown category gets all zeros - safe fallback.

In [3]:
from scipy.stats import ks_2samp

def detect_drift(X_train, X_new, feature_names, threshold=0.05):
    """
    Compare feature distributions between training and new data.
    Returns a dict of drifted features with their p-values.
    
    threshold: p-value below this means distributions are significantly different
    """
    drifted = {}
    
    for i, feature in enumerate(feature_names):
        stat, p_value = ks_2samp(X_train[:, i], X_new[:, i])
        
        if p_value < threshold:
            drifted[feature] = {
                'ks_statistic': round(stat, 4),
                'p_value': round(p_value, 4)
            }
    
    return drifted

**What ks_2samp does:** 

it compares two distributions by finding the maximum distance between their cumulative distribution functions (CDFs). 

It returns a statistic (how far apart they are) and a p-value (probability they came from the same distribution). Low p-value → distributions are different → drift.

In [4]:
np.random.seed(42)

# training data: 1000 players, 3 features
X_train = np.column_stack([
    np.random.normal(28, 3, 1000),    # age, mean 28
    np.random.normal(30, 2, 1000),    # speed, mean 30
    np.random.normal(75, 5, 1000)     # weight, mean 75
])

# new data: age and weight same, but speed has drifted
X_new = np.column_stack([
    np.random.normal(28, 3, 200),     # age, same
    np.random.normal(33, 2, 200),     # speed, mean shifted from 30 to 33
    np.random.normal(75, 5, 200)      # weight, same
])

features = ['age', 'speed_kmh', 'weight_kg']
result = detect_drift(X_train, X_new, features)
print(result)
# {'speed_kmh': {'ks_statistic': 0.58, 'p_value': 0.0}} — speed drifted

{'speed_kmh': {'ks_statistic': np.float64(0.517), 'p_value': np.float64(0.0)}}


**Why KS test and not something else?** → KS is non-parametric (no distribution assumptions), works on continuous features. For categorical features you'd use chi-squared test instead.

**What's the threshold?** → 0.05 is standard but in production with many features you'd apply Bonferroni correction (threshold / num_features) to avoid false alarms.

**What do you do when drift is detected?** → Alert, investigate root cause (data pipeline issue vs genuine shift), retrigger retraining if it's real, and never auto-retrain blindly without checking labels are still valid.

**Limitation?** → KS test needs enough samples to be reliable. With tiny batches it won't catch subtle drift.